# 01 -- Stage 1: Train on Image Datasets

Trains `ForensicMultiHeadNet` on `data/labels.csv` (CASIA2.0, CoMoFoD,
Coverage, IMD2020, FaceForensics++ images -- combine them into one
`labels.csv` + `images/`/`masks/` before running this). Saves
`models/image_model.pth`.

In [ ]:
import sys, os

IS_COLAB = os.path.exists("/var/colab/hostname")
IS_KAGGLE = (not IS_COLAB) and (os.path.exists("/kaggle/input") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ)
PROJECT_ROOT = "/kaggle/working/ForensicAI_Lab" if IS_KAGGLE else "/content/ForensicAI_Lab"

if not os.path.exists(PROJECT_ROOT):
    !git clone https://github.com/MRoy03/Forensic_LAB.git {PROJECT_ROOT}

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from notebooks.utils.colab_common import mount_drive, install_requirements, is_colab

if is_colab():
    mount_drive()
    install_requirements(PROJECT_ROOT)
elif IS_KAGGLE:
    !pip install -q -r requirements-train.txt
else:
    print("Not in Colab/Kaggle -- assuming requirements-train.txt is already installed locally.")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import pandas as pd
import numpy as np
import cv2

from config.loader import get_settings, resolve_dataset_root, resolve_path
from services.preprocessing.pipeline import preprocess_image
from services.detection.model import ForensicMultiHeadNet

settings = get_settings()
dataset_root = resolve_dataset_root(settings)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training on:", device)
print("Dataset root:", dataset_root)

if str(dataset_root) != str(Path(settings.data.root_drive)):
    print(
        "\n[WARNING] dataset_root is a LOCAL path, not your Drive path "
        f"({settings.data.root_drive}). If you prepared real data via 00b and "
        "expected it to be found here, this usually means that data was never "
        "copied to Drive (00b's 'persist to Drive' step) before this fresh "
        "session started. Proceeding will train on whatever labels.csv/images "
        "actually exist at the local path above -- double check that's really "
        "your prepared dataset and not an empty/placeholder one before continuing."
    )

## If running on Kaggle: pull in the unified dataset from an attached input

Kaggle has no Drive access, so `dataset_root` above falls back to this
session's local disk, which starts empty. This does **not** re-run
00b's per-source conversion (CASIA2/CoMoFoD/Coverage/IMD2020/FFpp each
have their own raw layout) -- it expects the **already-unified**
`labels.csv` + `images/` + `masks/` that 00b produces, attached as one
Kaggle Dataset (e.g. by uploading those three straight from Drive).
Skip this cell entirely on Colab.

In [ ]:
if IS_KAGGLE:
    import shutil

    def find_file(root, name):
        for dirpath, _, filenames in os.walk(root, followlinks=True):
            if name in filenames:
                return Path(dirpath) / name
        return None

    KAGGLE_INPUT_SLUG = "forensicai-lab-image-data"  # <-- change to your attached dataset's folder name
    kaggle_input_path = Path("/kaggle/input") / KAGGLE_INPUT_SLUG

    labels_csv_dst = dataset_root / settings.data.labels_csv
    # A header-only data/labels.csv ships in the repo itself (scaffolding
    # placeholder) -- a fresh `git clone` always creates it, so checking
    # mere existence would always skip the copy below. Check for actual rows.
    already_has_data = labels_csv_dst.exists() and len(pd.read_csv(labels_csv_dst)) > 0
    if already_has_data:
        print(f"{labels_csv_dst} already has data locally -- skipping copy from Kaggle input.")
    else:
        found_labels = find_file(kaggle_input_path, Path(settings.data.labels_csv).name)
        assert found_labels, (
            f"No {settings.data.labels_csv} found under {kaggle_input_path} -- "
            "check KAGGLE_INPUT_SLUG matches your attached dataset's folder name, "
            "and that it contains the unified images/masks/labels.csv output of "
            "00b (not the raw per-source zips -- those need 00b's own Option D "
            "conversion step first, not this notebook)."
        )
        unified_root = found_labels.parent
        print("Found unified dataset at:", unified_root)
        shutil.copy(found_labels, labels_csv_dst)
        for subdir_name in (settings.data.images_dir, settings.data.masks_dir):
            src_dir = unified_root / subdir_name
            if src_dir.exists():
                shutil.copytree(src_dir, dataset_root / subdir_name, dirs_exist_ok=True)
        print(f"Copied unified dataset from {unified_root} to {dataset_root}")

In [ ]:
class ForensicImageDataset(Dataset):
    def __init__(self, labels_csv, images_dir, masks_dir, image_size):
        self.df = pd.read_csv(labels_csv)
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.image_size = image_size
        self.label_to_idx = {l: i for i, l in enumerate(settings.labels.binary)}
        manip_classes = settings.labels.manipulation_types
        self.manip_to_idx = {m: i for i, m in enumerate(manip_classes)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_bgr = cv2.imread(str(self.images_dir / row["filename"]))
        if image_bgr is None:
            # Safety net: the validation cell below should have already removed
            # unreadable rows, but if any slip through (e.g. a transient Drive
            # I/O hiccup), skip gracefully instead of crashing the whole run.
            print(f"[WARN] Could not read {self.images_dir / row['filename']}, substituting another sample")
            return self[(idx + 1) % len(self)]
        x = preprocess_image(image_bgr, tuple(self.image_size))

        label_idx = self.label_to_idx[row["label"]]
        row_type = row["type"]
        # "none" (real/unmanipulated) isn't one of the 6 manipulation_types
        # classes, so .get() would silently default it to class 0 (copy_move)
        # and train the manip head against a wrong label on every real
        # sample. has_manip=0 excludes those samples from manip_loss instead
        # -- manip_idx is still a valid (if arbitrary) class index so the
        # loss function doesn't error, it's just zero-weighted.
        # Membership test (not == none_type) also catches other
        # non-manipulation-class values, e.g. the feedback pipeline's
        # "unknown" fallback for a fake sample with no specified type.
        has_manip = 1.0 if row_type in self.manip_to_idx else 0.0
        manip_idx = self.manip_to_idx.get(row_type, 0)

        mask_path = self.masks_dir / row["filename"]
        if mask_path.exists():
            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            mask = cv2.resize(mask, tuple(self.image_size))
            mask = (mask > 127).astype(np.float32)[np.newaxis, :, :]
            has_mask = 1.0
        else:
            mask = np.zeros((1, *self.image_size), dtype=np.float32)
            has_mask = 0.0

        return (
            torch.from_numpy(x),
            torch.tensor(label_idx, dtype=torch.long),
            torch.tensor(manip_idx, dtype=torch.long),
            torch.from_numpy(mask),
            torch.tensor(has_mask, dtype=torch.float32),
            torch.tensor(has_manip, dtype=torch.float32),
        )

## Validate the dataset before training

Raw dataset downloads commonly have a handful of missing/corrupt/
unreadable files (wrong codec, truncated copy, etc.). Without this check,
training would crash deep inside a DataLoader worker process the first
time it hits one -- possibly after a long wait -- with a cryptic OpenCV
error that doesn't even name the file. This finds and removes any bad
rows from `labels.csv` up front, with a clear list of what got dropped.

In [ ]:
labels_csv = dataset_root / settings.data.labels_csv
images_dir = dataset_root / settings.data.images_dir
masks_dir = dataset_root / settings.data.masks_dir

_check_df = pd.read_csv(labels_csv)
_bad_indices = []

# No progress output until this finishes otherwise -- on Drive-mounted
# images, thousands of individual cv2.imread calls can genuinely take
# several minutes, and with zero feedback that's indistinguishable from
# a hung cell. Print progress every 2000 rows so it's visibly alive.
for i, row in _check_df.iterrows():
    if i > 0 and i % 2000 == 0:
        print(f"  ...checked {i}/{len(_check_df)}")
    img_path = images_dir / row["filename"]
    if cv2.imread(str(img_path)) is None:
        _bad_indices.append(i)
        print(f"[UNREADABLE] {img_path}")

print(f"\n{len(_bad_indices)} of {len(_check_df)} rows failed to load.")

if _bad_indices:
    _cleaned_df = _check_df.drop(index=_bad_indices).reset_index(drop=True)
    _cleaned_df.to_csv(labels_csv, index=False)
    print(f"Removed {len(_bad_indices)} bad rows -- labels.csv now has {len(_cleaned_df)} rows.")
else:
    print("All rows loaded successfully -- no changes made.")

In [ ]:
dataset = ForensicImageDataset(labels_csv, images_dir, masks_dir, settings.inference.image_size)
loader = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=2)
print("Dataset size:", len(dataset))

In [ ]:
# pretrained=True loads ImageNet weights into the encoder (timm adapts
# the first conv layer to in_chans=5 automatically). Every run before
# this was trained from random init -- none of these notebooks actually
# passed `pretrained=` through, so it silently used the class default
# (False) regardless of this setting. A from-scratch CNN needs far more
# data than these datasets provide; this is likely the dominant reason
# results stayed near chance even after fixing sampling/loss weighting.
model = ForensicMultiHeadNet(
    backbone=settings.models.backbone,
    in_chans=settings.inference.input_channels,
    num_manip_classes=len(settings.labels.manipulation_types),
    pretrained=settings.models.pretrained_encoder,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
cls_loss_fn = torch.nn.CrossEntropyLoss()
manip_loss_fn = torch.nn.CrossEntropyLoss(reduction="none")
mask_loss_fn = torch.nn.BCEWithLogitsLoss(reduction="none")

NUM_EPOCHS = 5  # adjust based on dataset size / Colab session limits
checkpoint_path = resolve_path(settings.models.image_model_checkpoint)
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
# Saves after every epoch (unconditional overwrite, not a best-of --
# this notebook has no validation split to judge "best" by) so a quota
# cutoff or disconnect mid-run only costs the CURRENT epoch, not the
# whole session.
model.train()
for epoch in range(NUM_EPOCHS):
    running_loss = 0.0
    for x, label_idx, manip_idx, mask, has_mask, has_manip in loader:
        x, label_idx, manip_idx = x.to(device), label_idx.to(device), manip_idx.to(device)
        mask, has_mask, has_manip = mask.to(device), has_mask.to(device), has_manip.to(device)

        optimizer.zero_grad()
        out = model(x)

        loss_cls = cls_loss_fn(out["cls_logits"], label_idx)
        manip_loss_per_sample = manip_loss_fn(out["manip_logits"], manip_idx)
        loss_manip = (manip_loss_per_sample * has_manip).sum() / (has_manip.sum() + 1e-8)
        mask_loss_per_px = mask_loss_fn(out["mask_logits"], mask)
        loss_mask = (mask_loss_per_px.mean(dim=[1, 2, 3]) * has_mask).sum() / (has_mask.sum() + 1e-8)

        loss = loss_cls + loss_manip + loss_mask
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} -- loss: {running_loss / len(loader):.4f}")
    torch.save(model.state_dict(), checkpoint_path)
    print(f"  -> saved checkpoint to {checkpoint_path}")

## Persist the checkpoint to Drive

This session's local disk (`checkpoint_path` above) doesn't survive a
disconnect. If running on Colab, copy it to Drive now so Stage 3/6 (run
in their own fresh sessions later) can find it.

In [ ]:
import shutil

if IS_COLAB:
    drive_models_dir = Path("/content/drive/MyDrive/ForensicAI_Lab/models")
    drive_models_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy(checkpoint_path, drive_models_dir / checkpoint_path.name)
    print("Copied checkpoint to", drive_models_dir / checkpoint_path.name)
else:
    print("Not on Colab -- skipping Drive copy (e.g. on Kaggle, download it from the Output tab instead).")